In [26]:

from pathlib import Path

import pandas as pd

from morocco_retail_intelligence.data.cleaning import (
    remove_duplicates_with_report,
)
from morocco_retail_intelligence.data.loader import load_data
from morocco_retail_intelligence.data.profiling import (
    build_dataset_overview,
    profile_dataframe,
)
from morocco_retail_intelligence.data.transformations import (
    convert_age_to_int,
    convert_date_columns,
)
from morocco_retail_intelligence.data.validation import (
    validate_foreign_key,
    validate_primary_key,
)


In [27]:

data = load_data(Path("../data/raw"))

data.orders.head()



,order_id,customer_id,order_date,order_status,delivery_date,shipping_city,shipping_fee_mad
0,O00001,C0713,2024-04-09,Delivered,2024-04-13,Tangier,30.0
1,O00002,C0942,2025-11-19,Delivered,2025-11-23,Marrakech,25.0
2,O00003,C0191,2024-12-13,Delivered,2024-12-20,Agadir,25.0
3,O00004,C0364,2024-08-21,Delivered,2024-08-24,Casablanca,30.0
4,O00005,C0801,2024-11-22,Delivered,2024-11-23,Agadir,35.0


In [28]:

datasets = {
    "customers": data.customers,
    "products": data.products,
    "orders": data.orders,
    "order_items": data.order_items,
    "payments": data.payments,
    "reviews": data.reviews,
}


In [29]:

# for name, df in datasets.items():
#     print(f"\n{'-' * 50 } \n {'DATASET: ' + name.upper()} \n {'-' * 50}")

#     print("Shape:", df.shape)
#     print("Columns:", df.columns.tolist())
#     print("\nData types:\n" + str(df.dtypes))
#     print("\nMissing values:")
#     print(df.isna().sum().T)



In [30]:


# overview = build_dataset_overview(datasets)

# display(overview)

In [31]:
# orders_profile = profile_dataframe(data.orders)

# display(orders_profile)

In [32]:
# for name, df in datasets.items():
#     print(f"DATASET: {name.upper()}")
#     display(profile_dataframe(df))

In [33]:
# for name, df in datasets.items():
#     print(f"\n{name.upper()}")
#     display(df.dtypes.to_frame("data_type"))

In [34]:


customers = convert_date_columns(
    data.customers,
    ["signup_date"],
)

orders = convert_date_columns(
    data.orders,
    ["order_date", "delivery_date"],
)

reviews = convert_date_columns(
    data.reviews,
    ["review_date"],
)

In [35]:
# display(orders.dtypes.to_frame("data_type"))

In [36]:
orders["order_year"] = orders["order_date"].dt.year

orders["order_month"] = orders["order_date"].dt.month

orders["order_day"] = orders["order_date"].dt.day

orders["order_weekday"] = orders["order_date"].dt.day_name()

display(orders["order_year"].head(2))

0    2024
1    2025
Name: order_year, dtype: int32

In [37]:
# customers.dtypes


In [38]:
customers = convert_age_to_int(
    customers,
    ["customer_age"]
)

customers.dtypes

customer_id                    str
customer_age                 Int64
city                           str
customer_segment               str
signup_date         datetime64[us]
dtype: object

In [39]:
primary_keys = {
    "customers": ("customer_id", data.customers),
    "products": ("product_id", data.products),
    "orders": ("order_id", data.orders),
    "order_items": ("order_item_id", data.order_items),
    "payments": ("payment_id", data.payments),
    "reviews": ("review_id", data.reviews),
}

In [40]:
primary_key_report = pd.DataFrame([
    validate_primary_key(
        df=df,
        column=column,
        dataset_name=name,
    )
    for name, (column, df) in primary_keys.items()
])

display(primary_key_report)

,dataset,column,total_rows,missing_keys,duplicate_keys,is_valid
0,customers,customer_id,1000,0,0,True
1,products,product_id,100,0,0,True
2,orders,order_id,3205,0,5,False
3,order_items,order_item_id,5376,0,0,True
4,payments,payment_id,3200,0,0,True
5,reviews,review_id,2019,0,0,True


In [41]:
relationships = [
    (
        "orders → customers",
        data.orders,
        "customer_id",
        data.customers,
        "customer_id",
    ),
    (
        "order_items → orders",
        data.order_items,
        "order_id",
        data.orders,
        "order_id",
    ),
    (
        "order_items → products",
        data.order_items,
        "product_id",
        data.products,
        "product_id",
    ),
    (
        "payments → orders",
        data.payments,
        "order_id",
        data.orders,
        "order_id",
    ),
    (
        "reviews → orders",
        data.reviews,
        "order_id",
        data.orders,
        "order_id",
    ),
    (
        "reviews → customers",
        data.reviews,
        "customer_id",
        data.customers,
        "customer_id",
    ),
]

In [42]:
foreign_key_report = pd.DataFrame([
    validate_foreign_key(
        child_df=child_df,
        child_column=child_column,
        parent_df=parent_df,
        parent_column=parent_column,
        relationship_name=name,
    )
    for name, child_df, child_column, parent_df, parent_column
    in relationships
])

display(foreign_key_report)

,relationship,missing_foreign_keys,invalid_foreign_keys,is_valid
0,orders → customers,0,0,True
1,order_items → orders,0,0,True
2,order_items → products,0,0,True
3,payments → orders,0,0,True
4,reviews → orders,0,0,True
5,reviews → customers,0,0,True


In [43]:
invalid_orders = data.orders.loc[
    data.orders["customer_id"].notna()
    & ~data.orders["customer_id"].isin(
        data.customers["customer_id"]
    )
]

display(invalid_orders)

,order_id,customer_id,order_date,order_status,delivery_date,shipping_city,shipping_fee_mad


In [44]:
reviews.duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
2014    False
2015    False
2016    False
2017    False
2018    False
Length: 2019, dtype: bool

In [45]:
orders["order_id"].duplicated().sum()

np.int64(5)

In [46]:
orders_clean, orders_cleaning_report = (
    remove_duplicates_with_report(data.orders)
)

display(pd.DataFrame([orders_cleaning_report]))

,original_rows,cleaned_rows,removed_rows
0,3205,3200,5


In [48]:
orders_clean["order_id"].duplicated().sum()

np.int64(0)